# 10. 미니 RAG — 로컬 모델 vs API, 같은 질문 15개

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/11_mini_rag_local_vs_api.ipynb)

> 🧭 **스토리** — 드디어 규정 도우미 v1을 조립한다. 08번의 검색기(KURE-v1)로 근거를 찾고, LLM이 **근거 번호를 달아** 답한다. 답하는 LLM은 둘: 노트북 안에서 도는 **EXAONE 1.2B(Ollama)** 와 **gpt-6-luna(API)**. 같은 코드, 같은 질문 15개로 정확도·인용·'모른다'·지연·비용을 잰다.
>
> 📖 **이론 연결** — 이론2 p54~61(RAG 개요·파이프라인·프롬프트 조립·출처 검증·오류 진단·평가·로컬 vs API) · ⏱ **20분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
설치 → Ollama 서버 + EXAONE 모델(812MB) → API 키 → 클라이언트·비용 헬퍼 순서입니다. 04번에서 쓴 셀과 같습니다.
> 답 생성은 02번처럼 transformers로 EXAONE을 직접 띄워도 됩니다. 오늘은 API와 **같은 클라이언트 코드**로 비교하려고 Ollama를 씁니다.

In [ ]:
%pip -q install -U sentence-transformers transformers chromadb openai tiktoken

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-ai-api/data)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-ai-api/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ Ollama 설치 + 서버 실행 (코랩 리눅스 VM 안에서). 첫 실행 1분 안팎.
import subprocess, shutil, time, os, urllib.request
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# 서버를 백그라운드로 띄우고 준비될 때까지 기다립니다
if not os.environ.get("_OLLAMA_STARTED"):
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.environ["_OLLAMA_STARTED"] = "1"
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
print("Ollama 준비 완료:", subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
# ⚙️ 모델 받기 — GGUF(Q4_K_M 양자화) 를 허깅페이스에서 Ollama로 바로 당겨옵니다
MODEL_LOCAL = "hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M"
import subprocess, time
t0 = time.time()
r = subprocess.run(["ollama", "pull", MODEL_LOCAL], capture_output=True, text=True)
print(r.stdout[-300:] if r.returncode == 0 else r.stderr[-500:])
print(f"pull 소요 {time.time()-t0:.0f}초")
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.
print("키 입력 완료 · 모델:", MODEL_API)

In [ ]:
# ⚙️ 같은 클라이언트, 다른 백엔드 — base_url 만 바꾸면 로컬 모델 / OpenAI / (3차시) 네이버 CLOVA Studio 를 같은 코드로 부릅니다
from openai import OpenAI
BACKENDS = {
    "openai": dict(base_url=None, api_key=os.environ.get("OPENAI_API_KEY", "none"), model="gpt-6-luna"),
    "local":  dict(base_url="http://localhost:11434/v1", api_key="ollama", model="hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M"),
    # 3차시 예고: 네이버 CLOVA Studio 는 OpenAI 호환 엔드포인트를 제공합니다 (키는 3차시에 발급)
    "clova":  dict(base_url="https://clovastudio.stream.ntruss.com/v1/openai", api_key=os.environ.get("CLOVA_API_KEY", "none"), model="HCX-005"),
}
def make_client(backend: str):
    cfg = BACKENDS[backend]
    client = OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"]) if cfg["base_url"] else OpenAI(api_key=cfg["api_key"])
    return client, cfg["model"]

In [ ]:
# ⚙️ 비용 계산 — 2026-10 기준 공식 단가(USD per 1M tokens). https://developers.openai.com/api/docs/pricing
PRICES = {  # (입력, 캐시 입력, 출력)
    "gpt-6-luna":   (0.10, 0.01, 0.50),
    "gpt-5-nano":   (0.05, 0.005, 0.40),
    "gpt-5.6-luna": (0.20, 0.02, 1.20),
    "gpt-6-astra":  (10.0, 1.0, 50.0),
    "text-embedding-3-small": (0.02, 0.02, 0.0),
}
def cost_usd(model: str, in_tok: int, out_tok: int = 0, cached_tok: int = 0) -> float:
    p_in, p_cache, p_out = PRICES.get(model, (0.10, 0.01, 0.50))
    return ((in_tok - cached_tok) * p_in + cached_tok * p_cache + out_tok * p_out) / 1_000_000
def cost_krw(usd: float, rate: float = 1400.0) -> float:
    return usd * rate

## 1. 인덱스 만들기 — 청크 13개 → KURE-v1 → Chroma 🔵
08번에서 한국어 질문에 가장 강했던 `KURE-v1`로 청크를 임베딩하고, 벡터DB **Chroma**에 넣습니다.
- Chroma 기본 임베딩 함수는 영어용입니다. 그래서 **임베딩을 직접 계산해서 넣고**(`embeddings=`), 거리는 `cosine`으로 지정합니다(이론2 p44).
- `metadatas`에 장 번호·종류를 같이 넣어 두면 나중에 "제4장만 검색" 같은 필터를 걸 수 있습니다.

In [ ]:
import time, re, torch, numpy as np, pandas as pd, chromadb
from sentence_transformers import SentenceTransformer

chunks = load_json("hanbit_chunks_gt.json")
qa = load_json("hanbit_qa.json")
kure = SentenceTransformer("nlpai-lab/KURE-v1", device=DEVICE, model_kwargs={"dtype": torch.float32})

t0 = time.time()
emb = kure.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(f"청크 {len(chunks)}개 임베딩 {time.time()-t0:.1f}초 · 차원 {emb.shape[1]}")

db = chromadb.Client()                                  # 메모리 안의 임시 DB (런타임이 끝나면 사라짐)
try:
    db.delete_collection("hanbit")                      # 셀을 다시 실행해도 되도록
except Exception:
    pass
col = db.create_collection("hanbit", configuration={"hnsw": {"space": "cosine"}}, embedding_function=None)
col.add(ids=[c["id"] for c in chunks],
        documents=[c["text"] for c in chunks],
        embeddings=emb.tolist(),
        metadatas=[{"section": c["section"], "kind": c["kind"]} for c in chunks])
print("Chroma 컬렉션:", col.name, "· 문서 수", col.count())

## 2. 검색 — 질문 → 상위 k개 🔵

In [ ]:
def retrieve(q, k=5):
    qv = kure.encode(q, normalize_embeddings=True)
    r = col.query(query_embeddings=[qv.tolist()], n_results=k, include=["documents", "metadatas", "distances"])
    return [dict(id=i, text=d, meta=m, dist=x)
            for i, d, m, x in zip(r["ids"][0], r["documents"][0], r["metadatas"][0], r["distances"][0])]

for h in retrieve("과장의 국내 출장 일비는 하루 얼마인가?"):
    print(f"{h['id']:<20} 거리 {h['dist']:.3f} (유사도 {1-h['dist']:.3f}) · {h['meta']}")

# 검색 단계만 따로 평가: 근거 있는 13문항의 정답 청크가 상위 k개에 들어갔나 (08번과 같은 규칙)
def gold_ids(item):
    if item["type"] == "unanswerable":
        return set()
    text_id = f"s{item['section']}_text"
    if item["table"] is None:
        return {text_id}
    table_id = f"s{item['section']}_{item['table']}"
    return {table_id, text_id} if item["type"] == "table+text" else {table_id}
answerable = [q for q in qa if q["type"] != "unanswerable"]
def retrieval_hit(k, search=retrieve):
    return np.mean([bool({h["id"] for h in search(q["q"], k)} & gold_ids(q)) for q in answerable])
print(f"\n검색 hit@1 {retrieval_hit(1):.2f} · hit@5 {retrieval_hit(5):.2f}")

💡 **해석** — Chroma의 `cosine` 거리는 **1 − 코사인 유사도**입니다. 작을수록 가깝습니다. 강사 환경에서 1등 `s2_t1_domestic`의 거리는 0.321(유사도 0.68)이었습니다.
검색 hit@5가 1.00이면 **13문항 모두 정답 근거가 LLM에게 전달된다**는 뜻입니다. 이제부터 틀리는 답은 검색 탓이 아니라 **생성(LLM) 탓**입니다. RAG가 틀렸을 때 검색과 생성 중 어디서 틀렸는지 나눠 보는 습관이 중요합니다(이론2 p59).

## 3. 프롬프트 조립 🔵
이론2 p57의 원칙을 그대로 코드로 옮깁니다.
1. **근거는 위, 질문은 아래** — 긴 근거 뒤에 질문을 두면 모델이 질문을 잊지 않습니다.
2. **근거마다 번호** `[1]`, `[2]` … — 답에 번호를 달게 해서 **출처를 확인**할 수 있게 합니다.
3. **모르면 모른다고** — 근거에 없으면 정해진 문장으로 답하게 합니다. 환각을 줄이는 가장 싼 방법입니다.

In [ ]:
SYSTEM = ("당신은 한빛테크 사내 규정 도우미입니다. 아래 [근거]만 사용해 한두 문장으로 짧게 답하세요. "
          "근거에 없으면 '규정에서 찾을 수 없습니다'라고 답하고, 답할 때는 근거 번호를 [n] 형식으로 붙여라.")

def build_prompt(q, hits):
    ctx = "\n\n".join(f"[{i}] ({h['id']}) {h['text']}" for i, h in enumerate(hits, 1))
    return f"[근거]\n{ctx}\n\n[질문] {q}\n\n(답 끝에 사용한 근거 번호를 [1]처럼 붙이세요.)"

example = build_prompt(qa[0]["q"], retrieve(qa[0]["q"], k=2))
print(example)

## 4. 같은 함수, 두 백엔드 🔵
`make_client("local")`과 `make_client("openai")`는 같은 `OpenAI` 클라이언트이고 `base_url`만 다릅니다. 그래서 `chat()` 하나로 둘 다 부르고, `rag_answer()`는 검색 + 프롬프트 조립 + `chat()`입니다.
- 두 백엔드 모두 `reasoning_effort="none"`으로 **추론(생각) 단계를 끄고** 바로 답하게 합니다. 지연·비용을 같은 조건에서 비교하기 위해서입니다.
- EXAONE 4.0은 추론 모델이라 그냥 부르면 **생각 글을 먼저 길게 쓰고** `</think>` 뒤에 답을 씁니다. 혹시 생각 글이 섞여 나와도 답만 남도록 `</think>` 앞부분을 잘라 내는 안전장치도 둡니다.

💡 Ollama는 OpenAI 호환 `reasoning_effort`를 think on/off로 매핑한다 (0.35 기준 실측). 강사 환경 예비 측정에서 생각을 끄자 15문항이 64.6초에서 17.5초로 약 3.7배 빨라졌고, 정답 키워드 수는 11/13으로 같았습니다.

In [ ]:
LOCAL_THINK = False    # 🟡 True로 바꾸면 로컬 모델이 생각 단계를 켠 채 답합니다(느려짐). 결과를 비교해 보세요.

clients = {b: make_client(b) for b in ["local", "openai"]}

# API 키가 유효한지 한 번만 확인 (모델 정보 조회는 과금되지 않음). 키가 없으면 API 열은 건너뜁니다.
try:
    clients["openai"][0].models.retrieve(clients["openai"][1])
    OPENAI_OK = True
except Exception as e:
    OPENAI_OK = False
    print("OpenAI API 미사용(키 확인):", type(e).__name__, "— 로컬 모델만 비교합니다.")

def chat(backend, messages, max_out=None):
    """두 백엔드 공통 호출. 반환: (답 텍스트, usage, 지연초)"""
    client, model = clients[backend]
    kw = {}
    if backend == "openai":
        kw = dict(max_completion_tokens=max_out or 300, reasoning_effort="none")
    else:
        kw = dict(max_tokens=max_out or 600, temperature=0)
        if not LOCAL_THINK:
            kw["reasoning_effort"] = "none"       # Ollama는 이것을 think off로 처리
    t0 = time.time()
    r = client.chat.completions.create(model=model, messages=messages, **kw)
    text = (r.choices[0].message.content or "").split("</think>")[-1].strip()   # 안전장치: 생각 글 제거
    return text, r.usage, time.time() - t0

def rag_answer(backend, q, k=5, search=retrieve):
    hits = search(q, k)
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(q, hits)}]
    if backend == "openai" and not OPENAI_OK:
        return dict(text="API 미호출(키 없음)", in_tok=0, out_tok=0, latency=None, cost=None, hits=hits)
    try:
        text, u, latency = chat(backend, msgs)
    except Exception as e:
        return dict(text=f"호출 실패: {type(e).__name__} {str(e)[:80]}", in_tok=0, out_tok=0, latency=None, cost=None, hits=hits)
    cost = cost_usd(clients[backend][1], u.prompt_tokens, u.completion_tokens) if backend == "openai" else 0.0
    return dict(text=text, in_tok=u.prompt_tokens, out_tok=u.completion_tokens, latency=latency, cost=cost, hits=hits)

q = qa[0]["q"]
for b in ["local", "openai"]:
    a = rag_answer(b, q)
    lat = f"{a['latency']:.1f}초" if a["latency"] else "-"
    print(f"[{b}] {a['text']}\n   입력 {a['in_tok']} · 출력 {a['out_tok']} 토큰 · {lat}\n")

💡 **해석** — 강사 환경(Mac CPU)에서 로컬 EXAONE은 출력 67토큰, 0.4~1.1초 만에 "과장 직급의 일비는 **30,000원** … [1]"이라고 답했습니다. 생각 단계를 켜면 같은 질문에 출력이 516토큰, 6.6초로 늘었습니다(대부분 잘라 낸 생각 글).
API 열은 키가 있어야 채워집니다. 키가 없으면 `API 미호출(키 없음)`으로 표시하고 로컬만 비교합니다. gpt-6-luna는 보통 1초 안팎에 한두 문장으로 답합니다.

## 5. 15문항 전부 — 비교 표 🔵
질문 15개(표 조회 9 · 본문 3 · 계산 1 · 근거 없음 2)를 두 백엔드로 돌리고 다음을 잽니다(이론2 p60).
- **정답 키워드**: 답에 정답 키워드(예: `30,000`)가 들어 있나. 근거 있는 13문항 기준.
- **근거 인용**: 답에 `[1]` 같은 번호가 있나.
- **모른다**: 근거 없는 2문항에 "찾을 수 없다"류로 답했나.
- **지연·비용**: 문항당 평균 초, 15문항 총비용(로컬은 0원 — 대신 GPU·전기·운영 비용).

In [ ]:
UNKNOWN = re.compile(r"찾을 수 없|포함되어 있지 않|명시되어 있지 않|명시되지 않|언급되어 있지 않|언급되지 않|정보가 없|근거가 없|알 수 없")
CITE = re.compile(r"\[\d+\]")

results = {}
for b in ["local", "openai"]:
    t0 = time.time()
    results[b] = [rag_answer(b, q["q"]) for q in qa]
    print(f"{b}: 15문항 {time.time()-t0:.0f}초")

def evaluate(b):
    rs = results[b]
    if rs[0]["latency"] is None and rs[0]["in_tok"] == 0:
        return {"정답 키워드(13)": "-", "근거 인용(13)": "-", "모른다(2)": "-", "평균 지연(초)": "-", "총비용(USD)": "-"}
    ans = [(q, r) for q, r in zip(qa, rs) if q["keywords"]]
    unk = [(q, r) for q, r in zip(qa, rs) if q["type"] == "unanswerable"]
    lat = [r["latency"] for r in rs if r["latency"]]
    return {"정답 키워드(13)": f"{sum(any(k in r['text'] for k in q['keywords']) for q, r in ans)}/13",
            "근거 인용(13)": f"{sum(bool(CITE.search(r['text'])) for q, r in ans)}/13",
            "모른다(2)": f"{sum(bool(UNKNOWN.search(r['text'])) for q, r in unk)}/2",
            "평균 지연(초)": round(float(np.mean(lat)), 2),
            "총비용(USD)": round(sum(r["cost"] or 0 for r in rs), 6)}

display(pd.DataFrame({"로컬 EXAONE 1.2B Q4": evaluate("local"), "API gpt-6-luna": evaluate("openai")}).T)

side = pd.DataFrame({"유형": [q["type"] for q in qa],
                     "질문": [q["q"][:24] for q in qa],
                     "정답": [q["a"][:16] for q in qa],
                     "로컬": [r["text"].replace("\n", " ")[:70] for r in results["local"]],
                     "API": [r["text"].replace("\n", " ")[:70] for r in results["openai"]]}, index=[q["id"] for q in qa])
with pd.option_context("display.max_colwidth", 80):
    display(side)

💡 **해석** — 강사 환경(Mac CPU, Ollama, 생각 단계 끔) 측정값:

| 지표 | 로컬 EXAONE 1.2B Q4 | API gpt-6-luna |
|---|---|---|
| 정답 키워드(근거 있는 13문항) | 11/13 | 수업 때 키로 실측 |
| 근거 번호 인용 | 13/13 | 〃 |
| 근거 없는 2문항에 "모른다" | 2/2 (단, 1문항은 "다만…"으로 추측을 덧붙임) | 〃 |
| 문항당 평균 지연 | 0.4~0.7초 | 〃 |
| 15문항 총비용 | 0원 | 약 $0.002 추정(입력 약 1.7만·출력 약 1천 토큰) |

- **검색은 완벽했는데(hit@5 1.00) 생성에서 2문항을 틀렸습니다.**
  - Q3(차장의 유럽 일비): 표 2에서 '차장 이상' 열 대신 '과장 이하' 열을 읽어 60달러라고 답했습니다. 근거는 맞게 받았는데 **표의 열을 헷갈린** 것입니다.
  - Q13(과장 2박 3일 합계): 일비·숙박비 단가는 맞게 찾았지만 **곱하고 더하는 계산**에서 틀렸습니다.
- Q15(재택근무)는 "명시적으로 언급되지 않습니다"라고 한 뒤 "다만…"으로 다른 조항을 끌어와 추측을 붙였습니다. 생각 단계를 켠 예비 측정에서는 연차 표로 "약 22~23일"이라는 답을 **지어냈습니다**. 검색기는 관련 없는 질문에도 가장 가까운 청크를 **항상** 돌려주므로, '모른다'는 생성 모델이 판단해야 합니다.
- 질문 끝에 "(답 끝에 근거 번호를 붙이세요)"를 한 번 더 적은 덕분에 인용이 13/13 나왔습니다. 이 문장을 빼면 작은 모델은 인용을 거의 달지 않았습니다(예비 측정 0~1/13). **지시는 질문 가까이에 한 번 더**가 작은 모델에 잘 듣습니다.
- 생각 단계를 켜도(`LOCAL_THINK = True`) 정답 수는 11/13으로 같았고 지연만 약 7~10배(문항당 4.7초) 늘었습니다. 틀리는 문항만 Q3 대신 Q4로 바뀌었습니다.

## 6. 바꿔서 실행 🟡
### 6-1. k를 바꾸면? (2 / 5 / 10)
k가 작으면 근거를 놓치고, 크면 프롬프트가 길어져(비용·지연↑) 작은 모델이 헷갈립니다.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")
rows = []
for k in [2, 5, 10]:
    avg_tok = np.mean([len(enc.encode(SYSTEM + build_prompt(q["q"], retrieve(q["q"], k)))) for q in qa])
    row = {"k": k, "검색 hit@k": round(retrieval_hit(k), 2), "평균 입력 토큰(추정)": int(avg_tok),
           "API 1만 건 입력비(USD)": round(cost_usd("gpt-6-luna", avg_tok) * 10_000, 2)}
    if k != 5:     # k=5는 위에서 이미 돌렸음
        outs = [rag_answer("local", q["q"], k=k) for q in answerable]
    else:
        outs = [r for q, r in zip(qa, results["local"]) if q["type"] != "unanswerable"]
    row["로컬 정답 키워드"] = f"{sum(any(kw in o['text'] for kw in q['keywords']) for q, o in zip(answerable, outs))}/13"
    rows.append(row)
display(pd.DataFrame(rows).set_index("k"))

💡 **해석** — 강사 환경 측정값:

| k | 검색 hit@k | 평균 입력 토큰 | API 1만 건 입력비 | 로컬 정답 키워드 |
|---|---|---|---|---|
| 2 | 1.00 | 526 | $0.53 | 11/13 |
| 5 | 1.00 | 1,190 | $1.19 | 11/13 |
| 10 | 1.00 | 2,208 | $2.21 | 13/13 |

- 청크가 13개뿐이라 k=2에서도 정답 근거가 다 들어갔습니다. k를 늘리면 **입력 토큰과 비용은 k에 비례해 늘어납니다**(k=10은 k=2의 4.2배).
- 이번에는 k=10에서 로컬 모델이 13문항을 모두 맞혔습니다. 관련 청크(본문 + 표)를 함께 받자 열 혼동·계산 실수가 줄었습니다. 반대로 문서가 크면 관련 없는 청크가 섞여 작은 모델이 헷갈리기도 합니다. k는 **hit@k가 충분히 높아지는 가장 작은 값**에서 시작해 답 품질을 보고 조정합니다.

### 6-2. 청킹: 고정 200토큰 vs 헤딩(장·표) 단위
지금까지 쓴 청크는 장·표 경계에서 자른 **구조 기반 청크**입니다(08번 Docling 청킹과 같은 생각). 이번에는 규정집 마크다운을 **200토큰마다 기계적으로** 잘라 비교합니다.
고정 청크에는 id가 없으므로, 원문에서 **정답 청크와 30자 이상 겹치면** 적중으로 셉니다.

In [ ]:
import chromadb
md = load_text("hanbit_rules_gt.md")
spans = {}                                                   # 정답 청크가 마크다운의 어디에 있는지 (시작, 끝)
for c in chunks:
    body = c["text"].split("\n\n", 1)[1]
    s = md.find(body[:60]); spans[c["id"]] = (s, s + len(body))

toks = enc.encode(md)
mb = md.encode("utf-8")
def char_pos(n_tok):                                         # 토큰 n개까지의 글자 위치
    return len(mb[:len(enc.decode_bytes(toks[:n_tok]))].decode("utf-8", errors="ignore"))
fixed = []
for a in range(0, len(toks), 200):                           # 200토큰씩, 겹침 없이
    c0, c1 = char_pos(a), char_pos(min(a + 200, len(toks)))
    fixed.append(dict(id=f"fix{len(fixed):02d}", text=md[c0:c1], span=(c0, c1)))
print(f"규정집 {len(toks)}토큰 → 고정 청크 {len(fixed)}개 (헤딩 청크는 {len(chunks)}개)")
print("예) fix03:", repr(fixed[3]["text"][:90]))

def make_search(items, texts):
    """items 목록을 임베딩해 별도 컬렉션을 만들고 retrieve와 같은 모양의 검색 함수를 돌려준다"""
    name = f"tmp{abs(hash(texts[0])) % 10**8}"
    try: db.delete_collection(name)
    except Exception: pass
    c = db.create_collection(name, configuration={"hnsw": {"space": "cosine"}}, embedding_function=None)
    c.add(ids=[it["id"] for it in items], documents=texts, embeddings=kure.encode(texts, normalize_embeddings=True).tolist())
    by_id = {it["id"]: it for it in items}
    def search(q, k=5):
        r = c.query(query_embeddings=[kure.encode(q, normalize_embeddings=True).tolist()], n_results=k)
        return [by_id[i] for i in r["ids"][0]]
    return search

def span_hit(search, k):
    def overlap(a, b): return max(0, min(a[1], b[1]) - max(a[0], b[0]))
    return round(float(np.mean([any(overlap(it["span"], spans[g]) >= 30 for it in search(q["q"], k) for g in gold_ids(q))
                                for q in answerable])), 2)

for c in chunks: c["span"] = spans[c["id"]]
search_head = make_search(chunks, [c["text"] for c in chunks])
search_fixed = make_search(fixed, [f["text"] for f in fixed])
cmp = pd.DataFrame([{"청킹": "헤딩(장·표) 단위", **{f"hit@{k}": span_hit(search_head, k) for k in (1, 3, 5)}},
                    {"청킹": "고정 200토큰", **{f"hit@{k}": span_hit(search_fixed, k) for k in (1, 3, 5)}}]).set_index("청킹")
display(cmp)

💡 **해석** — 강사 환경에서 헤딩 청크는 hit@1 0.92, 고정 200토큰 청크는 0.77이었습니다(hit@3부터는 둘 다 1.00).
위 예시 `fix03`처럼 고정 청크는 **문장 한가운데에서 시작**합니다("km당 300원을…"). '자가 차량'이라는 핵심 단어는 앞 청크에 남았습니다. 표도 중간에서 잘리면 머리글(직급·일비)과 숫자가 다른 청크로 갈라집니다. 08번에서 Docling으로 **구조(장·표)를 살려 자른 이유**가 이것입니다(이론2 p51).

### 6-3. 간이 Contextual Retrieval — 청크 앞에 '어디 소속인지' 붙이기
Anthropic의 Contextual Retrieval은 청크마다 "이 조각은 문서의 어디에 속하는가"를 LLM으로 요약해 앞에 붙입니다(이론2 p53). 여기서는 LLM 대신 **장 제목**을 붙이는 간이판을 해 봅니다.
- 헤딩 청크는 이미 `제2장 국내 출장 — 표 1. …`으로 시작합니다. 이 머리말을 **떼고** 재 보면 붙인 효과를 거꾸로 알 수 있습니다.
- 고정 청크에는 머리말이 없으므로 시작 위치의 **장 제목을 붙여** 봅니다.

In [ ]:
heads = [(m.start(), m.group(0)[3:].strip()) for m in re.finditer(r"^## 제\d+장.*$", md, flags=re.M)]
def title_at(pos):
    t = heads[0][1]
    for st, name in heads:
        if st <= pos: t = name
    return t

head_plain = [c["text"].split("\n\n", 1)[1] for c in chunks]                       # 머리말 뗀 헤딩 청크
fixed_ctx = [f"[{title_at(f['span'][0])}] {f['text']}" for f in fixed]               # 장 제목 붙인 고정 청크
variants = {"헤딩 청크(머리말 있음, 원래)": search_head,
            "헤딩 청크(머리말 뗌)": make_search(chunks, head_plain),
            "고정 200토큰(원래)": search_fixed,
            "고정 200토큰 + 장 제목": make_search(fixed, fixed_ctx)}
display(pd.DataFrame([{"청크": n, **{f"hit@{k}": span_hit(s, k) for k in (1, 3, 5)}} for n, s in variants.items()]).set_index("청크"))
print("예) 장 제목 붙인 고정 청크:", repr(fixed_ctx[3][:80]))

💡 **해석** — 강사 환경 측정값:

| 청크 | hit@1 | hit@3 |
|---|---|---|
| 헤딩 청크(머리말 있음) | **0.92** | 1.00 |
| 헤딩 청크(머리말 뗌) | 0.85 | 1.00 |
| 고정 200토큰 | 0.77 | 1.00 |
| 고정 200토큰 + 장 제목 | 0.69 | 1.00 |

- 헤딩 청크에서 `제2장 국내 출장 — 표 1.` 같은 머리말을 떼자 hit@1이 0.92에서 0.85로 떨어졌습니다. **머리말이 곧 문맥**이고, 우리 청크에는 이미 붙어 있었던 것입니다.
- 고정 청크에 장 제목을 붙이자 오히려 한 문항이 나빠졌습니다(0.77→0.69). 장 경계에 걸친 청크에 **시작 위치의 장 제목 하나만** 붙여서 뒷부분 내용과 제목이 어긋난 탓입니다. 문항 하나가 0.08을 움직이는 작은 평가라는 점도 있습니다.
- 진짜 Contextual Retrieval은 LLM이 청크마다 "이 조각은 무엇에 관한 것인지"를 문장으로 써서 붙입니다. 기계적으로 제목만 붙이는 간이판은 **항상 좋아지지 않습니다**. 붙이기 전후를 hit@k로 재 보고 결정하세요.

### 🤔 생각해보기

**로컬 모델이 진 문항은 어떤 유형인가? API 비용이 정당화되는 지점은 어디인가?** 위 비교 표와 답 표를 보고 답해 보세요.

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- 강사 환경에서 로컬 1.2B 모델은 **표 조회·본문 질문**은 대부분 맞혔지만 **표의 열 혼동(과장 이하/차장 이상)** 과 **계산(2박 3일 합계)** 에서 틀렸습니다. 근거 없는 '재택근무' 질문에는 "없다"고 하면서도 추측을 덧붙였습니다. 지시 따르기·다단계 추론·'모른다' 판단이 작은 모델의 약점입니다. 공개 키로 API 열을 채워 같은 유형에서 차이가 나는지 확인하세요.
- **API가 정당화되는 지점**: 틀린 답의 비용이 큰 업무(돈·규정·법률), 계산·비교처럼 여러 근거를 엮는 질문, 인용 형식을 꼭 지켜야 하는 경우. gpt-6-luna는 15문항에 1원 미만이라 **정확도 1문항의 가치가 비용보다 훨씬 큽니다**.
- **로컬이 맞는 지점**: 데이터를 밖으로 보낼 수 없을 때(개인정보·사내 기밀), 호출량이 매우 많아 API 비용이 커질 때, 네트워크가 없는 현장. 이때는 검색을 잘 맞추고(hit@k), 프롬프트를 단순하게, 계산은 코드(13번 툴콜링)에 맡겨 약점을 메웁니다.
- 판단은 감이 아니라 **오늘처럼 같은 질문셋으로 잰 표**로 합니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- RAG = 검색(근거 찾기) + 생성(근거로 답하기). 틀리면 검색과 생성 중 어디서 틀렸는지 나눠 본다.
- 프롬프트 조립 원칙: 근거는 위·질문은 아래·근거 번호·모르면 모른다.
- `base_url`만 바꾼 같은 클라이언트로 로컬 모델과 API를 같은 질문셋에서 비교할 수 있다.
- 청킹·k·Contextual Retrieval은 정답이 정해진 설정이 아니다. hit@k로 재서 고른다.

> 다음 노트북: **12_openai_api — OpenAI API 제대로 쓰기 (구조화 출력·스트리밍·캐싱)**